# Step 1: Dataset Curation, Validation, and Split Integrity

This notebook is the notebook-first entrypoint for SoundPulse dataset ingestion. It covers source discovery, metadata normalization, audio inspection, split generation, leakage checks, and export of a canonical manifest that downstream notebooks consume.

Expected output:
- A curated manifest JSON under `ml/data/curated/`
- A split summary and leakage report
- Inputs compatible with the shared SoundPulse contracts

In [1]:
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'
SRC_ROOT = ML_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from soundpulse_ml.datasets import (
    DEFAULT_MANIFEST_PATH,
    DEFAULT_SOURCE_MAPPING_PATH,
    DEFAULT_SUMMARY_PATH,
    load_dataset_manifest,
    manifest_to_dataframe,
    summarize_dataset_manifest,
    summarize_dataset_manifest_frame,
    write_phase1_dataset_artifacts,
 )

print({
    'project_root': str(PROJECT_ROOT),
    'manifest_path': str(DEFAULT_MANIFEST_PATH),
    'summary_path': str(DEFAULT_SUMMARY_PATH),
})

{'project_root': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse', 'manifest_path': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_manifest.v1.json', 'summary_path': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_manifest_summary.csv'}


## Configure dataset sources

Update the metadata paths, audio roots, and label mappings below to match the copies of ESC-50, FSD50K, AudioSet subsets, and local Nigerian background recordings on your machine.

In [3]:
PHASE1_DATASET_NOTES = {
    'enabled_datasets': ['ESC-50', 'FSD50K'],
    'disabled_datasets': {
        'AudioSet': 'Excluded from phase 1 because the local download is incomplete.',
    },
    'background_strategy': 'Use the null background class from sound_data_classes.md via FSD50K labels for dishes and pots and pans, frying food, music, and mechanical fan.',
}

PHASE1_DATASET_NOTES

{'enabled_datasets': ['ESC-50', 'FSD50K'],
 'disabled_datasets': {'AudioSet': 'Excluded from phase 1 because the local download is incomplete.'},
 'background_strategy': 'Use the null background class from sound_data_classes.md via FSD50K labels for dishes and pots and pans, frying food, music, and mechanical fan.'}

In [4]:
output_paths = write_phase1_dataset_artifacts()
manifest = load_dataset_manifest(DEFAULT_MANIFEST_PATH)
curated_df = manifest_to_dataframe(manifest)

print({name: str(path) for name, path in output_paths.items()})
curated_df.head()

{'manifest': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_manifest.v1.json', 'summary': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_manifest_summary.csv', 'source_mapping': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_class_to_source_mapping.json'}


,dataset_name,source_name,audio_path,class_slug,split,clip_id,source_group_id,duration_ms,sample_rate_hz,relative_path
0,esc50,ESC-50,C:\Users\ZBOOK STUDIO G5\Documents\GitHub\Soun...,dog_barking,validation,esc50-1-100032-A-0,esc50::100032,5000,44100,ml/data/raw/esc_50_dataset/audio/1-100032-A-0.wav
1,esc50,ESC-50,C:\Users\ZBOOK STUDIO G5\Documents\GitHub\Soun...,door_knock,validation,esc50-1-101336-A-30,esc50::101336,5000,44100,ml/data/raw/esc_50_dataset/audio/1-101336-A-30...
2,esc50,ESC-50,C:\Users\ZBOOK STUDIO G5\Documents\GitHub\Soun...,door_knock,validation,esc50-1-103995-A-30,esc50::103995,5000,44100,ml/data/raw/esc_50_dataset/audio/1-103995-A-30...
3,esc50,ESC-50,C:\Users\ZBOOK STUDIO G5\Documents\GitHub\Soun...,door_knock,validation,esc50-1-103999-A-30,esc50::103999,5000,44100,ml/data/raw/esc_50_dataset/audio/1-103999-A-30...
4,esc50,ESC-50,C:\Users\ZBOOK STUDIO G5\Documents\GitHub\Soun...,dog_barking,validation,esc50-1-110389-A-0,esc50::110389,5000,44100,ml/data/raw/esc_50_dataset/audio/1-110389-A-0.wav


In [5]:
curated_df.groupby(['dataset_name', 'class_slug']).size().unstack(fill_value=0)

class_slug,background,clock_alarm,crying_baby,dog_barking,door_knock,door_wood_creaking,doorbell,fire_alarm,speech,water_running
dataset_name,,,,,,,,,,
esc50,0,40,40,40,40,40,0,40,0,0
fsd50k,15358,0,137,513,364,563,141,1543,2152,445


In [6]:
curated_df.groupby(['split', 'class_slug']).size().unstack(fill_value=0)

class_slug,background,clock_alarm,crying_baby,dog_barking,door_knock,door_wood_creaking,doorbell,fire_alarm,speech,water_running
split,,,,,,,,,,
test,2150,8,38,108,106,190,34,408,693,168
train,11995,24,114,376,248,351,90,1101,1318,237
validation,1213,8,25,69,50,62,17,74,141,40


## Group-aware split generation

We split by `source_group_id` to avoid train, validation, and test leakage across derivative clips from the same source event.

In [7]:
split_summary_df = summarize_dataset_manifest_frame(manifest)
split_summary_df

,dataset_name,split,class_slug,clip_count
0,esc50,test,clock_alarm,8
1,esc50,test,crying_baby,8
2,esc50,test,dog_barking,8
3,esc50,test,door_knock,8
4,esc50,test,door_wood_creaking,8
5,esc50,test,fire_alarm,8
6,esc50,train,clock_alarm,24
7,esc50,train,crying_baby,24
8,esc50,train,dog_barking,24
9,esc50,train,door_knock,24


In [8]:
group_split_counts = curated_df.groupby('source_group_id')['split'].nunique()
leakage_groups = group_split_counts[group_split_counts > 1]
assert leakage_groups.empty, f'Split leakage detected for groups: {list(leakage_groups.index)}'

print({
    'leakage_group_count': int(leakage_groups.shape[0]),
    'manifest_path': str(DEFAULT_MANIFEST_PATH),
})

{'leakage_group_count': 0, 'manifest_path': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_manifest.v1.json'}


In [9]:
with DEFAULT_SOURCE_MAPPING_PATH.open('r', encoding='utf-8') as handle:
    class_to_source_mapping = json.load(handle)

class_to_source_mapping

{'fire_alarm': {'esc50_categories': ['fireworks'],
  'fsd50k_mids': ['/m/07pp_mv']},
 'clock_alarm': {'esc50_categories': ['clock_alarm'], 'fsd50k_mids': []},
 'door_knock': {'esc50_categories': ['door_wood_knock'],
  'fsd50k_mids': ['/m/07r4wb8']},
 'doorbell': {'esc50_categories': [], 'fsd50k_mids': ['/m/03wwcy']},
 'crying_baby': {'esc50_categories': ['crying_baby'],
  'fsd50k_mids': ['/m/0463cq4']},
 'water_running': {'esc50_categories': [], 'fsd50k_mids': ['/m/02jz0l']},
 'dog_barking': {'esc50_categories': ['dog'], 'fsd50k_mids': ['/m/05tny_']},
 'door_wood_creaking': {'esc50_categories': ['door_wood_creaks'],
  'fsd50k_mids': ['/m/07q6cd_']},
 'speech': {'esc50_categories': [], 'fsd50k_mids': ['/m/09x0r']},
 'background': {'esc50_categories': [],
  'fsd50k_mids': ['/m/04brg2', '/m/0dxrf', '/m/04rlf', '/m/02x984l'],
  'note': 'Phase-1 Nigerian noise fallback built from the null background class combinations in sound_data_classes.md.'}}

In [10]:
print(summarize_dataset_manifest(DEFAULT_MANIFEST_PATH))
output_paths

SoundPulse dataset manifest loaded successfully
datasets=2
clips=21456
split_counts={'test': 3903, 'train': 15854, 'validation': 1699}
class_counts=background:15358,clock_alarm:40,crying_baby:177,dog_barking:553,door_knock:404,door_wood_creaking:603,doorbell:141,fire_alarm:1583,speech:2152,water_running:445


{'manifest': WindowsPath('C:/Users/ZBOOK STUDIO G5/Documents/GitHub/SoundPulse/ml/data/curated/soundpulse_manifest.v1.json'),
 'summary': WindowsPath('C:/Users/ZBOOK STUDIO G5/Documents/GitHub/SoundPulse/ml/data/curated/soundpulse_manifest_summary.csv'),
 'source_mapping': WindowsPath('C:/Users/ZBOOK STUDIO G5/Documents/GitHub/SoundPulse/ml/data/curated/soundpulse_class_to_source_mapping.json')}